# 03 - Calibration walkthrough

Runs the LangGraph calibration loop on a sim-as-real target and visualizes
convergence. The agent starts from the uncalibrated ISP defaults and recovers
a hidden 'real camera look'.

For reproducibility this notebook drives the loop with a *scripted* proposer
(no Ollama needed). A real run uses `make_llm_client()` instead, shown at the
end. Outputs cleared in version control.

In [ ]:
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from sensorforge.agent.graph import CalibrationContext, run_calibration
from sensorforge.agent.state import AgentState, TunableParams
from sensorforge.agent.tools import capture_real, render_sim
from sensorforge.isp.params import ISPParams
from sensorforge.sim.camera import SimCamera
from sensorforge.sim.renderer import SimRenderer

SCENE = Path.cwd().parent / "scenes" / "checkerboard.xml"

# The hidden target the agent must recover (a neutral webcam look).
HIDDEN = TunableParams(
    exposure_ms=11.0,
    black_level=0.04,
    awb_gain_r=1.15,
    awb_gain_g=1.0,
    awb_gain_b=1.25,
    gamma=2.0,
)

In [ ]:
class ScriptedLLM:
    """Stands in for the real LLM: walks the knobs from defaults toward the
    hidden target over a few steps so we get a convergence curve."""

    def __init__(self, start, target, fractions=(0.4, 0.7, 0.9, 1.0)):
        s, t = start.model_dump(), target.model_dump()
        self.proposals = [json.dumps({k: s[k] + f * (t[k] - s[k]) for k in t}) for f in fractions]

    def generate(self, messages):
        if "JSON only" in messages[-1].content:
            return self.proposals.pop(0) if self.proposals else "{}"
        return "the white balance and tone do not match the reference yet"

In [ ]:
cam = SimCamera.from_scene(SCENE)
with SimRenderer(cam) as r:
    linear = r.render()

base = ISPParams()
rng = np.random.default_rng(0)
real = capture_real(
    "sim", linear_rgb=linear, hidden_params=HIDDEN.apply_to(base), frames=16, rng=rng
)

run_dir = Path(tempfile.mkdtemp(prefix="sf_cal_"))
llm = ScriptedLLM(TunableParams(), HIDDEN)
ctx = CalibrationContext(linear, real, base, llm, str(run_dir), rng, n_average=16)
final = run_calibration(ctx, AgentState(max_iters=20, tolerance_de=3.0, run_dir=str(run_dir)))

print("stop reason:", final.stop_reason)
print("baseline deltaE2000:", round(final.history[0].metrics["deltaE2000"], 2))
print("best deltaE2000:    ", round(final.best.metrics["deltaE2000"], 2))

## Convergence

deltaE2000 should fall toward the tolerance line; SSIM should rise.

In [ ]:
iters = [h.iteration for h in final.history]
de = [h.metrics["deltaE2000"] for h in final.history]
ss = [h.metrics["SSIM"] for h in final.history]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(iters, de, "o-")
ax[0].axhline(3.0, color="r", ls="--", label="tolerance")
ax[0].set_xlabel("iteration")
ax[0].set_ylabel("deltaE2000")
ax[0].legend()
ax[1].plot(iters, ss, "o-", color="green")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel("SSIM")
plt.tight_layout()

In [ ]:
initial = render_sim(linear, TunableParams().apply_to(base), np.random.default_rng(1))
best = render_sim(linear, final.best.params.apply_to(base), np.random.default_rng(2))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, img, title in [
    (ax[0], initial, "initial (uncalibrated)"),
    (ax[1], best, "calibrated"),
    (ax[2], real, "real (target)"),
]:
    a.imshow(img)
    a.set_title(title)
    a.axis("off")
plt.tight_layout()

## The assumptions log

Every accepted parameter change is recorded with its justification, the
artifact a sim engineer would hand over.

In [ ]:
print((run_dir / "assumptions.md").read_text())

## Running for real

Swap the scripted proposer for an actual LLM and the rest is identical:

```python
from sensorforge.agent.llm import make_llm_client
llm = make_llm_client("ollama")  # or set SENSORFORGE_LLM=anthropic
```

Or from the command line:

```bash
sensorforge calibrate --target uniform --real-source sim --max-iters 20
```